#### Interactive Pick and Place with `ROBOTIS FFW SG2`

This notebook runs an interactive pick-and-place demo in MuJoCo. A user selects an object on the table, the robot observes the scene with its head camera, estimates 3D pick/place targets from RGB-D perception, and executes the motion with IK-based control.

Pipeline overview:
- Configure object assets, perception options, and camera look-at settings.
- Build a MuJoCo scene with the FFW-SG2 robot, table, sampled Objaverse objects, and a bowl.
- Arrange the scene, create the radio-button UI, and initialize the perception/controller APIs.
- Use OWLv2 + SAM2 once per pick request to segment the selected object and the bowl.
- Convert masked depth into 3D point clouds and send the resulting targets to the controller.
- Step the simulation and render debug overlays in the notebook loop.

The notebook keeps the runtime loop explicit so observation, perception, controller updates, simulation stepping, and rendering remain easy to inspect.


In [ ]:
from package.init_scripts.init_ipython_setup import *
from package.init_scripts.init_qt import *
from package.api.pick_and_place_core import *
from package.api.pick_and_place_utils import *
from package.qt import *
from package.utility import *
from PyQt5.QtWidgets import QApplication
app = QApplication.instance()


#### 0. Configuration

This section defines the experiment settings used by the rest of the notebook.

Key values:
- `objaverse_path`: root directory for Objaverse assets.
- `objaverse_easypick_name_list`: object categories to sample from.
- `n_obj`: number of pickable objects placed on the table.
- `bowl_item`: bowl asset used as the place target.
- `head_lookat`: world-frame point used to aim the robot head camera.

`segmentation_options` controls the OWLv2 + SAM2 perception path. OWLv2 proposes text-conditioned bounding boxes, and SAM2 produces masks from those prompts. The current settings use a positive point from the selected bbox, avoid negative bbox points, clip masks to the detection box, and allow a bbox-mask fallback when SAM2 returns an empty mask.

`sample_obj_pairs(...)` samples concrete `(category, item_id)` pairs from the requested categories. These pairs determine which object MJCF files are merged into the scene.


In [ ]:
objaverse_path = '../asset/objaverse'
objaverse_easypick_name_list = ['bottled_water', 'can']
n_obj = 2
bowl_item = 'bowl_5'
bowl_body_name = 'body_obj_%s' % bowl_item
head_lookat = np.array([0.80, 0.0, 0.7])
segmentation_options = dict(
    det_threshold           = 0.1,
    fallback_det_thresholds = (0.05, 0.02),
    topk_per_query          = 1,
    neg_margin              = 10,
    use_bbox_neg_points     = False,
    fill_holes              = True,
    keep_largest            = True,
    prefer_pos_mask         = False,
    clip_mask_to_box        = True,
    box_clip_margin         = 4,
    bbox_mask_fallback      = True,
    overlay_alpha           = 0.5,
    use_text_aliases        = True,
    plot_seg                = True,
    plot_overlay_bright     = 0.5,
)

obj_pairs = sample_obj_pairs(root_dir  = objaverse_path,
                             obj_names = objaverse_easypick_name_list,
                             n_sample  = n_obj,
                             seed      = 12345,
                             verbose   = True)


#### 1. Build XML Model

This section builds the MuJoCo scene XML. The scene includes:
- the FFW-SG2 robot,
- a table and floor,
- sampled Objaverse objects,
- and a bowl used as the place target.

`build_pick_and_place_xml(...)` merges the required MJCF files and returns a `MuJoCoParser` environment. The returned `env` is used by the runtime cell for object placement, RGB-D capture, simulation stepping, and rendering.


In [ ]:
env = build_pick_and_place_xml(obj_pairs      = obj_pairs,
                               n_obj          = n_obj,
                               bowl_item      = bowl_item,
                               objaverse_path = objaverse_path,
                               verbose        = False)


#### 2. Runtime Setup and Main Loop

This section contains the full interactive runtime. The code is intentionally kept in one place so the execution order is visible.

The setup part performs the following steps:
- Create MuJoCo body names for the sampled objects.
- Group SG2 joints into arm, gripper, lift, and upper-body control sets.
- Place the objects and bowl on the table and initialize the robot posture.
- Build the radio-button UI for selecting the target object and triggering pick execution.
- Load the OWLv2 + SAM2 segmenter.
- Create `PickAndPlaceObservationAPI` for RGB-D/state/point-cloud observations.
- Create `InteractivePickAndPlaceLoopController` for mode/phase control and IK-based step commands.
- Open the MuJoCo viewer.

The loop then repeats while the viewer is alive:
1. Read the current UI state.
2. Capture a lightweight observation for rendering and controller updates.
3. When `pick` is turned on, run segmentation once for the selected object and once for the bowl.
4. Convert each mask and depth image into a point-cloud observation.
5. Send the pick/place centers to the controller.
6. Update the controller and step MuJoCo with the returned control command.
7. Render object frames, selected points, point clouds, IK targets, and the camera overlay.

The controller mode progresses through `idle`, `choose_to_pick`, `execute`, and `done`. During execution, it advances through the pick-and-place phases internally.


In [ ]:
# Arrange
obj_body_names = ["body_obj_%s" % item for _, item in obj_pairs]
joint_groups = get_sg2_joint_groups(env)
init_q = get_sg2_init_q(env, head_lookat=head_lookat)
obj_xyzs, bowl_xyz = arrange_pick_and_place_scene(
    env            = env,
    obj_pairs      = obj_pairs,
    obj_body_names = obj_body_names,
    bowl_body_name = bowl_body_name,
    n_obj          = n_obj,
)
env.forward(init_q, joint_names=joint_groups["use_whole_joint_names"])

# Radio button to select objects to pick
target_labels, label_to_idx = get_object_selection_options(
    env            = env,
    obj_pairs      = obj_pairs,
    obj_body_names = obj_body_names,
    reverse        = True,
)
selector = MultiRadioQtWidget(
    title          = "Pick Target",
    label_texts    = ["target", "pick"],
    option_texts   = [target_labels, ["off", "on"]],
    initial_texts  = [target_labels[0] if len(target_labels) > 0 else None, "off"],
    x_offset       = 0.0,
    y_offset       = 0.00,
    window_width   = 0.35,
    window_height  = 0.10,
    fontsize       = 20,
)

# Segmentation
segmenter = make_owlv2_sam2_segmenter(owl_local_dir='../hf_models/owlv2',
                                      sam2_local_dir='../hf_models/sam2',
                                      verbose=True)

# Observation class
obs_api = PickAndPlaceObservationAPI(
    env          = env,
    cam_name     = "head_cam",
    head_lookat  = head_lookat,
    joint_names  = joint_groups["use_upper_body_joint_names"],
    segmenter    = segmenter,
    segmentation_kwargs = segmentation_options,
)

# Interactive pick and place class
controller = InteractivePickAndPlaceLoopController(
    env                    = env,
    upper_body_joint_names = joint_groups["use_upper_body_joint_names"],
    gripper_joint_names    = joint_groups["use_gripper_joint_names"],
    idle_tick_goal         = 30,
    idle_nstep             = 10,
    execute_nstep          = 30,
    ik_err_thr             = 5e-2,
    hold_tick_goal         = 30,
    release_tick_goal      = 30,
    max_done_tick          = 10,
    grasp_pitch_deg        = 0.0,
    grasp_axis_len         = 0.1,
)

# Init viewer
env.init_viewer(title="FFW SG2 Interactive Pick and Place", width=1.0, height=1.0, fontscale=200)
env.viewer.set_transparency(transparent=False)
env.viewer.set_cam_info(180, 3.2, -30, [0.02, 0.01, 0.4])
perception_cache = None
last_selected_label = None

# Main loop
while env.is_viewer_alive():
    app.processEvents()

    selected_label = selector.get_button_option("target")
    pick_state = selector.get_button_option("pick")
    pick_requested = (pick_state == "on")
    selected_idx = label_to_idx[selected_label]
    selected_body_name = obj_body_names[selected_idx]

    # Basic observation
    obs = obs_api.get_observation(
        image        = True,
        pcd          = False,
        robot_state  = True,
        object_state = True,
        segmentation = False,
        extra_body_names = [bowl_body_name],
    )

    if selected_label != last_selected_label and controller.mode in ["idle", "choose_to_pick"]:
        perception_cache = None
        controller.reset(mode=controller.mode, clear_targets=True)
        last_selected_label = selected_label
        selector.set_button_option("pick", "off")
        pick_requested = False

    # Run OWLv2 + SAM2 only once after the user turns pick on.
    if pick_requested and perception_cache is None and controller.mode == "choose_to_pick":
        obj_name, obj_id = obj_pairs[selected_idx]
        obs_api.set_target_object(
            body_name  = selected_body_name,
            text_query = obj_name,
            obj_id     = obj_id,
            label      = selected_label,
        )
        pick_obs = obs_api.get_observation(
            image        = True,
            pcd          = False,
            robot_state  = True,
            object_state = True,
            segmentation = False,
            extra_body_names = [bowl_body_name],
        )
        pick_result = run_prompt_to_segmentation(
            segmenter      = segmenter,
            rgb            = pick_obs["image"]["rgb"],
            prompt         = obj_name,
            env            = env,
            obj_pairs      = obj_pairs,
            obj_body_names = obj_body_names,
            obj_idx        = selected_idx,
            **segmentation_options,
        )
        pick_pcd = build_point_cloud_observation(
            env              = env,
            depth            = pick_obs["image"]["depth"],
            mask             = pick_result["mask"],
            p_ego            = pick_obs["image"]["p_ego"],
            p_trgt           = pick_obs["image"]["p_trgt"],
            rsz_rate_for_pcd = 1 / 2,
            fovy             = 45,
            to_world         = True,
        )
        place_result = run_prompt_to_segmentation(
            segmenter = segmenter,
            rgb       = pick_obs["image"]["rgb"],
            prompt    = "round bowl",
            **segmentation_options,
        )
        place_pcd = build_point_cloud_observation(
            env              = env,
            depth            = pick_obs["image"]["depth"],
            mask             = place_result["mask"],
            p_ego            = pick_obs["image"]["p_ego"],
            p_trgt           = pick_obs["image"]["p_trgt"],
            rsz_rate_for_pcd = 1 / 2,
            fovy             = 45,
            to_world         = True,
        )
        perception_cache = dict(
            obs          = pick_obs,
            pick_result  = pick_result,
            pick_results = pick_result["candidates"],
            place_results = place_result["candidates"],
            place_result = place_result,
            pick_pcd     = pick_pcd,
            place_pcd    = place_pcd,
            pick_center  = pick_pcd["center"],
            place_center = place_pcd["center"],
        )
        # Set target
        controller.set_target_object(
            body_name = selected_body_name,
            obj_name  = obj_name,
            obj_id    = obj_id,
            label     = selected_label,
        )
        controller.set_pick_place_targets(
            pick_position  = perception_cache["pick_center"],
            place_position = perception_cache["place_center"],
        )

    rollout_info = controller.update(
        observation        = obs,
        selected_body_name = selected_body_name,
        pick_requested     = pick_requested,
    )

    # Update simulation from rollout_info.
    step_info = rollout_info["step"]
    if step_info["do_step"]:
        env.compensate_gravity(root_body_names=step_info["compensate_root_body_names"])
        env.step(ctrl=step_info["ctrl"],
                 joint_names=step_info["joint_names"],
                 nstep=step_info["nstep"])
        controller.after_step()

    # Render from rollout_info.
    render_info = rollout_info["render"]
    env.viewer_text_overlay("Mode", rollout_info["mode"])
    if rollout_info["phase"] is not None:
        env.viewer_text_overlay("Phase", rollout_info["phase"])
    env.viewer_text_overlay("Pick", pick_state)
    env.viewer_text_overlay("Selected", selected_label)

    for body_name in obj_body_names + [bowl_body_name]:
        env.plot_T(T=env.get_T(body_name, "body"), axis_len=0.05)
    env.plot_sphere(p=env.get_p(selected_body_name, type="body"), r=0.025, rgba=(1, 0.2, 0.1, 0.8))

    if perception_cache is not None:
        pick_points = perception_cache["pick_pcd"]["masked"]
        place_points = perception_cache["place_pcd"]["masked"]
        for p in pick_points[::max(1, len(pick_points)//300)]:
            env.plot_sphere(p=p, r=0.002, rgba=(1, 1, 0, 1))
        for p in place_points[::max(1, len(place_points)//300)]:
            env.plot_sphere(p=p, r=0.002, rgba=(0, 1, 1, 1))
        env.plot_sphere(p=perception_cache["pick_center"], r=0.012, rgba=(1, 0, 0, 1))
        env.plot_sphere(p=perception_cache["place_center"], r=0.015, rgba=(0, 0, 1, 1))
        env.plot_sphere(p=controller.targets["place"]["tcp"], r=0.012, rgba=(0, 1, 0, 1))
    if rollout_info["mode"] == "execute":
        render_info["ik_api"].render(env, r=0.01, alpha=0.5, scale_radius_by_weight=True)
    env.plot_T(T=obs["image"]["T_cam"], axis_len=0.05, axis_width=0.005)
    env.viewer.plot_rgb_overlay(obs["image"]["rgb"], loc="top right")
    env.plot_time()
    env.render()

    if rollout_info["status"] == "done":
        break

selector.close()
final_rgb_img = getattr(env, "final_rgb_img", None)
if final_rgb_img is not None:
    imshow(final_rgb_img, fig_w=3, title="Final scene", title_fs=8)
env.close_viewer()
